# From Threat Intelligence to Detection
## A Practical Notebook Workflow
Carlo Anez Mazurco • Cause-Finder / IgniteCyber Academy

All case content and telemetry are synthetic. Event command strings are data and are never executed. Live generation uses the OpenAI API; replay is explicitly labeled instructor-authored, with no local model.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/cti_lab').exists())
sys.path.insert(0, str(ROOT / 'src'))
from dotenv import load_dotenv
load_dotenv(ROOT / '.env', override=False)
from cti_lab import investigation as inv
from cti_lab import detection, nova_scan, assistant
from IPython.display import display, Markdown
print('AI mode:', os.getenv('AI_MODE', 'replay'))
print('Library versions:', inv.versions())

## 1. The intelligence requirement
Did behavior resembling the report occur here, and which detection conditions can we test? Read the numbered passages before making conclusions.

In [ ]:
sources = inv.load_sources()
display(Markdown((ROOT / 'data/reports/R01.md').read_text()))

## 2. MSTICPy extraction and normalization
The real MSTICPy extractor runs here. No silent regex fallback. Preserve source IDs and raw observables.

In [ ]:
iocs = inv.extract_iocs(sources)
display(iocs.head(20))
print('Extractor:', iocs.attrs['engine'])

## 3. Local enrichment
Enrichment adds context. A reputation label alone does not establish compromise.

In [ ]:
enriched = inv.enrich(iocs)
display(enriched.head(20))

## 4. Telemetry and timeline
Use field-aware matches, stable event IDs, and explicit time bounds. Several indicators can match the same event.

In [ ]:
events = inv.load_events()
hits = inv.correlate(iocs, events)
timeline = inv.timeline(events, host='WKSTN-1042')
display(hits.head(15))
print('Unique correlated events:', hits.EventID.nunique())
display(timeline.head(15))

## 5. ATT&CK evidence
Mappings are transparent analyst-authored hypotheses supported by event IDs. They do not come from keyword counting or actor attribution.

In [ ]:
mappings = inv.attack_evidence(events)
display(mappings)

## 6. Executable Sigma detections
Convert actual Sigma YAML through pySigma and run the generated SQL against SQLite. Compare broad and tuned PowerShell rules. ApprovedChange comes from the independent change inventory.

In [ ]:
results = detection.evaluate_all(events)
display(results['metrics'])
for name, sql in results['queries'].items():
    print(name, '\n', sql, '\n')

## 7. Changed infrastructure
The indicator rule misses rotated-task.invalid. The behavior rule still identifies its corresponding PowerShell events. This is fixture coverage, not a production efficacy benchmark.

In [ ]:
display(events[events.EventID.isin(['PX-A06','EP-A06'])])
print('Infrastructure matches:', results['matches']['infrastructure'])
print('Behavior matches:', results['matches']['powershell_tuned'])

## 8. Retrieved content is untrusted
Inspect the poisoned addendum as evidence. Its embedded instructions are text-only specimens. Nova records matches and policy separately from source preservation.

In [ ]:
poisoned = [s for s in sources if s['document_id']=='R06']
display(inv.pd.DataFrame([{'source_id':s['source_id'], **nova_scan.scan(s['text'])} for s in poisoned]))

## 9. Nova evaluation
View raw pattern matches separately from analyst-intent labels. Quotation handling is an application policy, not a promise that Nova understands every context. The held-out paraphrase intentionally demonstrates a blind spot.

In [ ]:
nova_results = nova_scan.evaluate_cases()
display(nova_results['metrics'])
display(nova_results['cases'].head(12))

## 10. Evidence-grounded OpenAI assistance
Live mode requires OPENAI_API_KEY and internet. Replay never contacts a model and is visibly labeled instructor-authored. Live failures raise an error; there is no silent fallback. No tools or commands are granted to the model.

In [ ]:
draft = assistant.analyze(mode=os.getenv('AI_MODE','replay'), question='Summarize Obsidian evidence, mappings, detections and gaps.')
print('PROVENANCE:', draft['provenance'])
display(Markdown(assistant.render(draft)))

## 11. Export the analyst handoff
Separate observations, hypotheses, and gaps. Retain the raw sources and cite exact passages. Rules stay experimental until reviewed in a real environment.

In [ ]:
outputs = inv.export_case(events, iocs, enriched, hits, mappings, results, nova_results, draft)
print('Wrote:', outputs)
display(Markdown((ROOT / 'outputs/analyst_brief.md').read_text()))

## Presenter close
The outcome is a repeatable investigation and detection test. Ask what evidence is missing before claiming attribution, successful exfiltration, or complete AI protection.